In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve

In [ ]:
DATA_DIR = Path(r"Z:\Users\predicting recovery\Lung")
lung_path = DATA_DIR / "lung.csv"
lung = pd.read_csv(lung_path, sep=None, engine="python", dtype={"UPN": str})
print(lung.shape)

In [ ]:
lung["compl"] = lung["compl"].replace(9, np.nan)
cohort = lung[lung["reggroep"] == 1].copy()
print(cohort.shape)
print(cohort["compl"].value_counts(dropna=False))

In [ ]:
comorbidity_cols = [
    c for c in cohort.columns
    if c.startswith("com") and not c.startswith("comp")
    and cohort[c].nunique(dropna=True) <= 5
]
cohort["n_comorbidities"] = cohort[comorbidity_cols].apply(
    lambda r: (r.fillna(0) > 0).sum(), axis=1
)
print(len(comorbidity_cols), "comorbidity flag columns")
print(cohort["n_comorbidities"].describe())

In [ ]:
cohort["BMI"] = cohort["gewicht"] / (cohort["lengte"] / 100) ** 2
cohort["asascore"] = cohort["asascore"].replace(9, np.nan)
cohort["ecog"] = cohort["ecog"].replace(9, np.nan)

In [ ]:
num_features = ["Leeftijd", "BMI", "asascore", "ecog", "n_comorbidities"]
cat_features = ["geslacht"]
target = "compl"

model_cols = num_features + cat_features + [target]
data = cohort[model_cols].dropna().copy()
data[target] = data[target].astype(int)
print(data.shape)
print(data[target].value_counts())

In [ ]:
n_events = int(data[target].sum())
n_params = len(num_features) + data["geslacht"].astype(str).nunique()
print(f"Complete cases: {len(data)}")
print(f"Events: {n_events}")
print(f"Approx parameters: {n_params}")
print(f"Events per variable: {n_events / n_params:.1f}")

In [ ]:
X = data[num_features + cat_features]
y = data[target]

pre = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(sparse=False, handle_unknown="ignore"), cat_features),
])
model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(solver="liblinear", max_iter=1000)),
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(model, X, y, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
auc = roc_auc_score(y, proba)
print(f"Cross-validated AUC: {auc:.3f}")

In [ ]:
frac_pos, mean_pred = calibration_curve(y, proba, n_bins=10, strategy="quantile")
cal_table = pd.DataFrame({"mean_predicted": mean_pred, "observed": frac_pos})
print(cal_table)

eps = 1e-6
p = np.clip(proba, eps, 1 - eps)
logit = np.log(p / (1 - p))
cal_lr = LogisticRegression(solver="liblinear")
cal_lr.fit(logit.reshape(-1, 1), y)
cal_slope = cal_lr.coef_[0][0]
print(f"Calibration slope: {cal_slope:.3f}")

In [ ]:
model.fit(X, y)
ohe_names = list(
    model.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(cat_features)
)
feat_names = num_features + ohe_names
coefs = model.named_steps["clf"].coef_[0]
or_table = pd.DataFrame({
    "feature": feat_names,
    "coefficient": coefs,
    "odds_ratio": np.exp(coefs),
})
print(or_table)

In [ ]:
output_dir = Path(r"Z:\Users\predicting recovery\AmanDeep\Data Analysis\lung_analysis\output_files")
output_dir.mkdir(parents=True, exist_ok=True)

results = pd.DataFrame({
    "metric": ["auc_cv", "calibration_slope", "n", "events", "epv"],
    "value": [auc, cal_slope, len(y), n_events, n_events / n_params],
})
results.to_csv(output_dir / "lung_baseline_model_results.csv", index=False)
print(results)